In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Import libraries
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import joblib

RANDOM_STATE = 42

In [ ]:
# 1) Load data
train_path = "/kaggle/input/titanic/train.csv"
test_path = "/kaggle/input/titanic/test.csv"
if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError("/kaggle/input/titanic/test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)


In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
# 2) Feature engineering functions
import pandas as pd
def extract_title(name: str) -> str:
    # Extract title from name and map rare titles to 'Rare'
    try:
        t = name.split(',')[1].split('.')[0].strip()
    except Exception:
        return "Unknown"
    mapping = {
        'Mlle':'Miss','Ms':'Miss','Mme':'Mrs',
        'Lady':'Rare','Countess':'Rare','Sir':'Rare','Jonkheer':'Rare',
        'Don':'Rare','Dona':'Rare','Capt':'Rare','Col':'Rare','Dr':'Rare','Rev':'Rare','Major':'Rare'
    }
    return mapping.get(t, t)

def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    
    # Title:
    out['Title'] = out['Name'].astype(str).apply(extract_title)
    # Family size and IsAlone
    out['FamilySize'] = out['SibSp'].fillna(0).astype(int) + out['Parch'].fillna(0).astype(int) + 1
    out['IsAlone'] = (out['FamilySize'] == 1).astype(int)
    # Ticket group size (count of identical tickets)
    out['TicketGroupSize'] = out.groupby('Ticket')['Ticket'].transform('count').fillna(1).astype(int)
    # Deck from Cabin first letter; fill missing with 'Unknown'
    out['Deck'] = out['Cabin'].fillna('Unknown').astype(str).str[0]
    # Log fare (handle zeros and missing)
    out['Fare'] = out['Fare'].fillna(0)
    out['LogFare'] = np.log1p(out['Fare'])
    # Keep useful columns; leave others so pipeline can select columns
    return out


In [ ]:
# 3) Columns to use 
num_features = ['Age','SibSp','Parch','LogFare','FamilySize','TicketGroupSize']
cat_features = ['Sex','Pclass','Embarked','Title','Deck','IsAlone']

In [ ]:

# Numeric transformer: median imputation
numeric_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical transformer: most frequent imputation + one-hot encode
categorical_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer([
    ('num', numeric_tf, num_features),
    ('cat', categorical_tf, cat_features)
], remainder='drop')  # drop other columns

In [ ]:
# 4) Full pipeline with feature engineering included via FunctionTransformer
from sklearn.preprocessing import FunctionTransformer
fe_transformer = FunctionTransformer(add_domain_features, validate=False)

# Logistic Regression pipeline (baseline)
pipe_lr = Pipeline([
    ('fe', fe_transformer),
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Random Forest pipeline (stronger model)
pipe_rf = Pipeline([
    ('fe', fe_transformer),
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))
])


In [ ]:
# 5) Quick CV evaluation
X = train.drop(columns=['Survived'])
y = train['Survived']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Running quick cross-val for baseline Logistic Regression...")
scores_lr = cross_val_score(pipe_lr, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("LogReg CV accuracy: %.4f +/- %.4f" % (scores_lr.mean(), scores_lr.std()))

print("Running quick cross-val for Random Forest...")
scores_rf = cross_val_score(pipe_rf, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV accuracy: %.4f +/- %.4f" % (scores_rf.mean(), scores_rf.std()))


In [ ]:
# 6) Hyperparameter tuning example: LogisticRegression (GridSearch over C)
print("Starting GridSearchCV for LogisticRegression (example)...")
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1.0, 5.0],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}
gs_lr = GridSearchCV(pipe_lr, param_grid=param_grid_lr, cv=50, scoring='accuracy', n_jobs=-1, verbose=1)
gs_lr.fit(X, y)
print("Best LR params:", gs_lr.best_params_)
print("Best LR CV acc:", gs_lr.best_score_)
best_lr = gs_lr.best_estimator_

In [ ]:
# 7) Hyperparameter tuning example: RandomForest
print("Starting GridSearchCV for RandomForest (ex)...")
param_grid_rf = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5]
}
gs_rf = GridSearchCV(pipe_rf, param_grid=param_grid_rf, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
gs_rf.fit(X, y)
print("Best RF params:", gs_rf.best_params_)
print("Best RF CV acc:", gs_rf.best_score_)
best_rf = gs_rf.best_estimator_

In [ ]:
#8) Choose final model (based on CV)
# Compare best LR vs best RF
print("Comparing best models on CV score:")
print("LR CV:", gs_lr.best_score_)
print("RF CV:", gs_rf.best_score_)

final_model = best_rf if gs_rf.best_score_ >= gs_lr.best_score_ else best_lr
print("Selected final model:", type(final_model.named_steps['clf']).__name__)

In [ ]:
# 9) Refit final model on full training data
final_model.fit(X, y)

In [ ]:
# 10) Prepare test set and predict
preds = final_model.predict(test)  # pipeline applies FE + preprocessing internally

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': preds.astype(int)
})
submission_file = "submission.csv"
submission.to_csv(submission_file, index=False)
print("Saved submission to", submission_file)

In [ ]:
print(submission.head())

In [ ]:
# 11) Save final model for later use

import joblib
joblib.dump(final_model, "titanic_final_pipeline.joblib")